# STAR-v3 — Train thử 10K trên Kaggle (X-VLM + LoRA)

**Cài đặt notebook (bắt buộc):** Settings → Accelerator = **GPU T4** · Internet = **ON**.

Notebook này: clone repo → dựng env pinned cho X-VLM (transformers 4.12.5, KHÔNG đụng torch CUDA của Kaggle) → tải checkpoint X-VLM 16M (825MB) → **Ô DATA** (cắm manifest của team data, hoặc fallback data tổng hợp 10K để chạy ngay) → sanity overfit → train full → evaluate.

Ước tính trên T4 (batch 16 @384): **~9–16 phút/epoch**, early-stop thường 3–5 epoch → **tổng ~45–100 phút** *(ước lượng — log in s/step thật ngay phút đầu)*.

In [ ]:
import os, pathlib
WORK = "/kaggle/working"
os.chdir(WORK)
if not pathlib.Path("star").exists():
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir(f"{WORK}/star")
print("repo:", os.getcwd())

In [ ]:
# ---- deps pinned cho X-VLM (KHONG dung den torch CUDA cua Kaggle) ----
!pip -q install "numpy<2" "scipy==1.11.4" "huggingface_hub==0.10.1" "tokenizers>=0.13,<0.14" sacremoses ruamel.yaml gdown pyarrow
!pip -q install --no-deps "transformers==4.12.5" "timm==0.4.9"

# patch 1: noi long pin tokenizers cua transformers (X-VLM dung tokenizer python rieng)
import importlib.util, pathlib, re
spec = importlib.util.find_spec("transformers")
f = pathlib.Path(spec.origin).parent / "dependency_versions_table.py"
f.write_text(re.sub(r'"tokenizers":[^,]+', '"tokenizers": "tokenizers"', f.read_text()))
print("patched:", f)

# X-VLM source + patch 2: CIDEr (captioning, khong dung) -> optional
if not pathlib.Path("third_party/X-VLM").exists():
    !git clone -q --depth 1 https://github.com/zengyan-97/X-VLM third_party/X-VLM
u = pathlib.Path("third_party/X-VLM/utils/__init__.py"); s = u.read_text()
old = "from utils.cider.pyciderevalcap.ciderD.ciderD import CiderD"
if "try:\n    " + old not in s:
    u.write_text(s.replace(old, "try:\n    " + old + "\nexcept Exception:\n    CiderD = None"))
print("X-VLM source ready")

# checkpoint X-VLM 16M (825MB, tai 1 lan)
pathlib.Path("data/checkpoints").mkdir(parents=True, exist_ok=True)
if not pathlib.Path("data/checkpoints/xvlm_16m_base.th").exists():
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
!ls -lh data/checkpoints/

## 📦 Ô DATA — TEAM DATA SỬA Ô NÀY
Khi manifest thật sẵn sàng (schema xem README §4 *Data contract*): upload thành **Kaggle Dataset** (gồm `manifest.parquet` + ảnh `.webp`), Add Data vào notebook, rồi đặt `USE_REAL_DATA = True` + sửa 2 đường dẫn.

Chưa có data thật → cell tự sinh **10K dữ liệu tổng hợp** (ảnh nhiễu + caption mẫu) — chỉ để **test pipeline/đo thời gian**, mAP không có ý nghĩa chất lượng.

In [ ]:
# ============================ DATA CELL (EDIT HERE) ============================
USE_REAL_DATA = False
MANIFEST   = "/kaggle/input/YOUR-DATASET/manifest.parquet"   # <-- sua khi co data that
IMAGE_ROOT = "/kaggle/input/YOUR-DATASET"                    # <-- sua khi co data that

if not USE_REAL_DATA:
    import numpy as np, pandas as pd
    from PIL import Image
    root = pathlib.Path("data/smoke10k"); (root / "img").mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    ACTIONS = ["falling", "running", "fighting", "lying", "throwing", "jumping", "pushing", "fainting"]
    N_UNIQUE, N_ROWS = 2500, 10000      # 2500 anh unique x4 -> decode cost that, sinh nhanh (~3-6 phut)
    def mk(name):
        p = root / "img" / name
        if not p.exists():
            Image.fromarray((rng.random((384, 384, 3)) * 255).astype("uint8")).save(p, format="WEBP", quality=80)
        return f"img/{name}"
    rows = []
    for i in range(N_UNIQUE):
        a = ACTIONS[i % len(ACTIONS)]
        path = mk(f"tr_{i}.webp")
        for _ in range(N_ROWS // N_UNIQUE):
            rows.append(dict(image_path=path, caption=f"a person {a} in area {i}", split="train",
                             sequence_id=f"seq{i}", scene=f"scene{i % 200}", action=a, bbox=None, keypoints=None))
        if i % 500 == 0:
            print("gen", i, flush=True)
    for s in range(300):                 # VAL-B queries
        a = ACTIONS[s % len(ACTIONS)]
        rows.append(dict(image_path=mk(f"va_{s}.webp"), caption=f"a person {a} on the street {s}",
                         split="valb", sequence_id=f"vseq{s}", scene=f"vscene{s}", action=a, bbox=None, keypoints=None))
    for d in range(700):                 # VAL-B distractors (anh-only, caption rong)
        rows.append(dict(image_path=mk(f"di_{d}.webp"), caption="", split="valb",
                         sequence_id=f"dist{d}", scene=f"dscene{d}", action="none", bbox=None, keypoints=None))
    pd.DataFrame(rows).to_parquet(root / "manifest.parquet", index=False)
    MANIFEST, IMAGE_ROOT = str(root / "manifest.parquet"), str(root)
    print(f"synthetic manifest: {len(rows)} rows")
print("MANIFEST =", MANIFEST)

In [ ]:
# Gate: overfit-one-batch (~3-5 phut) — pipeline PHAI hoi tu truoc khi chay full
!python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch \
    --set data.manifest={MANIFEST} data.image_root={IMAGE_ROOT}

In [ ]:
# TRAIN FULL 10K — theo doi dong log s/step dau tien de biet toc do thuc
import time
t0 = time.time()
!python scripts/train.py --config configs/star_v3_10k_kaggle.yaml \
    --set data.manifest={MANIFEST} data.image_root={IMAGE_ROOT} train.out_dir=/kaggle/working/run10k
print(f"TOTAL: {(time.time() - t0) / 60:.1f} min")

In [ ]:
# EVALUATE checkpoint tot nhat tren VAL-B (mAP / MRR / R@K)
!python scripts/evaluate.py --config configs/star_v3_10k_kaggle.yaml \
    --ckpt /kaggle/working/run10k/best.pth \
    --set data.manifest={MANIFEST} data.image_root={IMAGE_ROOT}
!ls -lh /kaggle/working/run10k/

## Ghi chú
- **Thời gian (ước lượng, T4):** batch 16 @384 ≈ 0.8–1.5 s/step → 625 step/epoch ≈ 9–16 phút/epoch; early-stop 3–5 epoch → **~45–100 phút**. VRAM dư thì thử `train.batch_size=24` (thêm vào `--set`).
- **Tham số 10K** nằm ở `configs/star_v3_10k_kaggle.yaml` (batch 16, fp16, 6 epoch + early-stop patience 2, eval mỗi 0.5 epoch, lr_lora 2e-4, λ₂=0.3).
- **Data thật:** chỉ cần sửa Ô DATA (flag + 2 path). Schema manifest: README §4. Distractor của VAL-B = row ảnh-only với `caption=""`.
- **Scale lên 100K:** đổi manifest + tăng `optim.epochs` trần lên 8–10; cân nhắc sweep `loss.lambda_smooth_ap` ∈ {0, 0.1, 0.3, 1.0}.
- ⚠️ Tổ hợp transformers 4.12.5 + torch mới của Kaggle đã được thiết kế tương thích (xbert tự chứa) nhưng **chưa chạy thử trên Kaggle** — nếu lỗi import ở cell train, báo lại stack trace.